In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
# 1. Install all Python Machine Learning, API, and UI libraries
!pip install CatBoost pandas numpy scikit-learn imbalanced-learn matplotlib seaborn xgboost lightgbm scipy optuna shap pyvis networkx streamlit fastapi uvicorn pydantic joblib

# 2. Install LocalTunnel (Required ONLY if using Colab to host the web app)
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.1 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
changed 22 packages in 2s
⠹
⠹3 packages are looking for funding
⠹  run `npm fund` for details
⠹

In [38]:
# =============================================================================
#  UPI MULE ACCOUNT DETECTION — COMPETITION PIPELINE (Colab / Full Env)
#  Requires: xgboost lightgbm catboost optuna shap imbalanced-learn sklearn
#  Install:  pip install xgboost lightgbm catboost optuna shap imbalanced-learn
# =============================================================================
import os, sys, time, json, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import scipy.stats as ss
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
from collections import Counter
from pathlib import Path

# ── sklearn ───────────────────────────────────────────────────────────────────
from sklearn.model_selection   import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing     import QuantileTransformer, OrdinalEncoder, StandardScaler
from sklearn.impute            import SimpleImputer
from sklearn.feature_selection import SelectFromModel
from sklearn.inspection        import permutation_importance
from sklearn.ensemble          import (RandomForestClassifier, GradientBoostingClassifier,
                                        IsolationForest, ExtraTreesClassifier,
                                        HistGradientBoostingClassifier)
from sklearn.linear_model      import LogisticRegression
from sklearn.neural_network    import MLPClassifier
from sklearn.neighbors         import NearestNeighbors
from sklearn.calibration       import CalibratedClassifierCV
from sklearn.metrics           import (classification_report, confusion_matrix,
                                        roc_auc_score, f1_score, accuracy_score,
                                        precision_score, recall_score,
                                        roc_curve, precision_recall_curve,
                                        average_precision_score)
# ── heavy libs ────────────────────────────────────────────────────────────────
import xgboost  as xgb
import lightgbm as lgb
import catboost
from catboost import CatBoostClassifier
from imblearn.under_sampling import TomekLinks
from imblearn.over_sampling  import SMOTE
import shap
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ═════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION
# ═════════════════════════════════════════════════════════════════════════════
CFG = dict(
    data_path     = '/content/drive/MyDrive/DataSet.csv',  # ← change this
    output_dir    = './mule_outputs',
    target_col    = 'F3924',
    random_seed   = 42,
    test_size     = 0.15,
    val_size      = 0.10,
    na_thresh     = 0.80,
    max_features  = 200,
    n_folds       = 5,
    optuna_trials = 50,      # set 0 to skip HPO and use defaults
    min_recall    = 0.0,
    smote_ratio   = 0.30,
    shap_min_pct  = 0.01,
    iso_n_est     = 200,
)
Path(CFG['output_dir']).mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(CFG['random_seed'])

def section(t): print(f"\n{'═'*65}\n  {t}\n{'═'*65}")
def log(m):     print(f"  {m}")
def save_fig(fig, name):
    p = f"{CFG['output_dir']}/{name}"
    fig.savefig(p, dpi=150, bbox_inches='tight'); plt.close(fig)
    log(f"[saved] {p}")

  # ═════════════════════════════════════════════════════════════════════════════
#  MODULE 1 — LOAD, VALIDATE & SANITIZE (AUTO-LEAK GUARD)
# ═════════════════════════════════════════════════════════════════════════════
section("MODULE 1 · LOAD, VALIDATE & SANITIZE")
t0 = time.time()

# 1. Load Data
df = pd.read_csv(CFG['data_path'], low_memory=False)
if 'Unnamed: 0' in df.columns: df = df.drop(columns=['Unnamed: 0'])
df = df.drop_duplicates()
log(f"Initial Shape: {df.shape[0]:,} rows × {df.shape[1]:,} cols")

# 2. 🛡️ Automated Leakage Guard
log("[Leak Guard] Scanning for Target Leaks (>85% correlation)...")
numeric_cols = df.select_dtypes(include=['number']).columns
correlations = df[numeric_cols].corrwith(df[CFG['target_col']]).abs()

LEAK_THRESH = 0.85
leaks = correlations[(correlations >= LEAK_THRESH) & (correlations.index != CFG['target_col'])].index.tolist()

if leaks:
    log(f"🚨 CRITICAL ALERT: Detected {len(leaks)} target leaks!")
    for col in leaks:
        log(f"   ↳ Banning '{col}' (Correlation: {correlations[col]:.4f})")
    df = df.drop(columns=leaks)
    log(f"✅ Sanitized Shape: {df.shape[0]:,} rows × {df.shape[1]:,} cols")
else:
    log("✅ Scan clean. No direct target leaks detected.")

# 3. Define Core Variables for the rest of the Pipeline
y_raw       = df[CFG['target_col']].astype(int)
X_raw       = df.drop(columns=[CFG['target_col']])
fraud_rate  = y_raw.mean()
n_fraud     = (y_raw == 1).sum()
n_legit     = (y_raw == 0).sum()
class_ratio = n_legit / max(n_fraud, 1)   # Crucial for scale_pos_weight

log(f"Fraud: {n_fraud:,} ({fraud_rate*100:.2f}%)  |  Legit: {n_legit:,}  |  Ratio: {class_ratio:.0f}:1")
# ═════════════════════════════════════════════════════════════════════════════
#  MODULE 2 — FEATURE ENGINEERING
#  Domain features from: Sahu et al. 2026 Table 5.1, Patil SSRN, IJISRT 2025
# ═════════════════════════════════════════════════════════════════════════════
section("MODULE 2 · FEATURE ENGINEERING")
X_eng = X_raw.copy()
for c in X_eng.columns: X_eng[c] = pd.to_numeric(X_eng[c], errors='coerce')

engineered = {}
num_cols   = X_eng.select_dtypes(include=[np.number]).columns.tolist()

def safe_ratio(a, b, fill=0.0):
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where(b != 0, a / b, fill)

# ── 1. Null density per row (sparse history = new/mule account) ───────────────
engineered['FE_null_density']       = X_eng.isnull().mean(axis=1)
engineered['FE_null_gt50']          = (X_eng.isnull().mean(axis=1) > 0.5).astype(float)
engineered['FE_n_nonzero']          = (X_eng.fillna(0) != 0).sum(axis=1)

# ── 2. Block-level mean/std (feature groups = time windows) ──────────────────
col_nums = [(int(c.replace('F','')), c) for c in X_eng.columns
            if c.startswith('F') and c[1:].isdigit()]
col_nums.sort()
for start in range(0, 4000, 100):
    block = [c for n, c in col_nums if start <= n < start+100]
    if len(block) < 5: continue
    sub = X_eng[block]
    engineered[f'FE_blk{start}_mean'] = sub.mean(axis=1)
    engineered[f'FE_blk{start}_std']  = sub.std(axis=1)

# ── 3. Key fraud-indicator interactions ──────────────────────────────────────
vol_feats = [f'F{i}' for i in range(3796, 3814) if f'F{i}' in X_eng.columns]
if vol_feats:
    engineered['FE_vol_mean'] = X_eng[vol_feats].mean(axis=1)
    engineered['FE_vol_max']  = X_eng[vol_feats].max(axis=1)
    engineered['FE_vol_cv']   = safe_ratio(X_eng[vol_feats].std(axis=1),
                                            X_eng[vol_feats].mean(axis=1).abs() + 1e-9)

if 'F3912' in X_eng.columns:
    engineered['FE_f3912_flag'] = X_eng['F3912'].fillna(0)
    if vol_feats:
        engineered['FE_f3912_x_vol'] = (X_eng['F3912'].fillna(0) *
                                         X_eng[vol_feats].mean(axis=1).fillna(0))

netflow_feats = [f'F{i}' for i in range(3835, 3838) if f'F{i}' in X_eng.columns]
if netflow_feats:
    engineered['FE_netflow_mean'] = X_eng[netflow_feats].mean(axis=1)
    engineered['FE_netflow_neg']  = (X_eng[netflow_feats].mean(axis=1) < 0).astype(float)

# ── 4. IQR-based row anomaly score ───────────────────────────────────────────
if len(num_cols) >= 5:
    num_arr = X_eng[num_cols[:100]].fillna(0).values
    q75, q25 = np.percentile(num_arr, 75, axis=0), np.percentile(num_arr, 25, axis=0)
    iqr = q75 - q25 + 1e-9
    engineered['FE_row_outlier_score'] = (
        np.abs((num_arr - np.median(num_arr, axis=0)) / iqr).mean(axis=1))

for k, v in engineered.items():
    X_eng[k] = v
log(f"Engineered {len(engineered)} new features  →  total: {X_eng.shape[1]}")

# ═════════════════════════════════════════════════════════════════════════════
#  MODULE 3 — PREPROCESSING
#  QuantileTransformer chosen per Ruchay et al. (Mathematics 2023):
#  highest IBA=0.9306 among ALL scalers on CreditCardFraud benchmark
# ═════════════════════════════════════════════════════════════════════════════
section("MODULE 3 · PREPROCESSING")
X = X_eng.copy().apply(pd.to_numeric, errors='coerce')

# Encode categoricals
cat_cols = X.select_dtypes(include=['object','category']).columns.tolist()
enc = None
if cat_cols:
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X[cat_cols] = enc.fit_transform(X[cat_cols].fillna('__MISSING__').astype(str))
    log(f"Encoded {len(cat_cols)} categorical cols")

# Drop high-null features
na_rates   = X.isnull().mean()
drop_na    = na_rates[na_rates > CFG['na_thresh']].index.tolist()
X          = X.drop(columns=drop_na)
log(f"Dropped {len(drop_na)} high-null cols (>{CFG['na_thresh']*100:.0f}%)")

# Median impute
imputer = SimpleImputer(strategy='median')
X_imp   = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)

# Drop zero-variance
var_mask = X_imp.var() >= 1e-6
X_imp    = X_imp.loc[:, var_mask]
log(f"Dropped {(~var_mask).sum()} zero-variance cols  →  {X_imp.shape[1]} features remain")

# QuantileTransformer
qt = QuantileTransformer(output_distribution='normal',
                          n_quantiles=min(1000, len(X_imp)),
                          random_state=CFG['random_seed'])
X_scaled = pd.DataFrame(qt.fit_transform(X_imp), columns=X_imp.columns, index=X_imp.index)
log("QuantileTransformer fitted (normal output)")

# ═════════════════════════════════════════════════════════════════════════════
#  MODULE 4 — STRATIFIED SPLIT  (test set locked before any resampling)
# ═════════════════════════════════════════════════════════════════════════════
section("MODULE 4 · STRATIFIED SPLIT")
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_scaled, y_raw, test_size=CFG['test_size'], stratify=y_raw,
    random_state=CFG['random_seed'])
X_train, X_thresh, y_train, y_thresh = train_test_split(
    X_trainval, y_trainval, test_size=CFG['val_size'], stratify=y_trainval,
    random_state=CFG['random_seed'])
log(f"Train={len(X_train):,} fraud={y_train.sum()} | Thresh={len(X_thresh):,} | Test={len(X_test):,} fraud={y_test.sum()}")

# ═════════════════════════════════════════════════════════════════════════════
#  MODULE 5 — TOMEK LINKS + SMOTE
#  Per Ruchay et al. 2023: "Tomek links removes boundary noise;
#  SMOTE synthesises minority samples" — jointly best strategy
# ═════════════════════════════════════════════════════════════════════════════
section("MODULE 5 · TOMEK LINKS + SMOTE  (imblearn)")
X_tr_np, y_tr_np = X_train.values, y_train.values
log(f"Before: legit={Counter(y_tr_np)[0]:,}  fraud={Counter(y_tr_np)[1]:,}")

tl          = TomekLinks(n_jobs=-1)
X_c, y_c   = tl.fit_resample(X_tr_np, y_tr_np)
log(f"After Tomek: legit={Counter(y_c)[0]:,}  fraud={Counter(y_c)[1]:,}")

smote       = SMOTE(random_state=CFG['random_seed'], k_neighbors=5,
                    sampling_strategy=CFG['smote_ratio'])
X_res, y_res = smote.fit_resample(X_c, y_c)
log(f"After SMOTE: legit={Counter(y_res)[0]:,}  fraud={Counter(y_res)[1]:,}")

# ═════════════════════════════════════════════════════════════════════════════
#  MODULE 6 — FEATURE SELECTION
#  Stage 1: RF SelectFromModel (max 200 features — not 50)
#  Stage 2: SHAP importance pruning (keep ≥ 1% of max SHAP)
# ═════════════════════════════════════════════════════════════════════════════
section("MODULE 6 · FEATURE SELECTION  (RF + SHAP)")
log("Stage 1 — RandomForest SelectFromModel …")
feat_rf = RandomForestClassifier(n_estimators=200, max_depth=15, max_features='sqrt',
                                  class_weight='balanced', random_state=CFG['random_seed'], n_jobs=-1)
feat_rf.fit(X_res, y_res)
selector     = SelectFromModel(feat_rf, prefit=True, max_features=CFG['max_features'])
X_tr_sel     = selector.transform(X_res)
X_thr_sel    = selector.transform(X_thresh.values)
X_te_sel     = selector.transform(X_test.values)
X_tv_sel     = selector.transform(X_trainval.values)
feat_names   = X_imp.columns[selector.get_support()].tolist()
log(f"Stage 1 kept: {len(feat_names)} features")

log("Stage 2 — SHAP pruning (XGBoost TreeExplainer) …")
shap_xgb = xgb.XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,          # FIX: was 0.2 in original
    scale_pos_weight=class_ratio,                  # FIX: missing in original
    tree_method='hist', eval_metric='auc',
    use_label_encoder=False, n_jobs=-1, random_state=CFG['random_seed'])
shap_xgb.fit(X_tr_sel, y_res, verbose=False)

sample_idx    = RNG.choice(len(X_tr_sel), min(3000, len(X_tr_sel)), replace=False)
explainer     = shap.TreeExplainer(shap_xgb)
shap_vals     = explainer.shap_values(X_tr_sel[sample_idx])
mean_shap     = np.abs(shap_vals).mean(axis=0)
shap_mask     = mean_shap >= mean_shap.max() * CFG['shap_min_pct']
X_tr_sel      = X_tr_sel[:,  shap_mask]
X_thr_sel     = X_thr_sel[:, shap_mask]
X_te_sel      = X_te_sel[:,  shap_mask]
X_tv_sel      = X_tv_sel[:,  shap_mask]
feat_names    = [f for f, k in zip(feat_names, shap_mask) if k]
log(f"Stage 2 kept: {len(feat_names)} features (SHAP pruned)")

# ═════════════════════════════════════════════════════════════════════════════
#  MODULE 7 — OPTUNA HPO
#  Tunes XGBoost & LightGBM via 3-fold CV on resampled training set
#  Bug fixes vs. pasted script: colsample_bytree [0.5,1.0] not [0.0,0.2],
#  scale_pos_weight included in both default and tuned params
# ═════════════════════════════════════════════════════════════════════════════
section(f"MODULE 7 · OPTUNA HPO  ({CFG['optuna_trials']} trials each)")

def tune_xgb():
    if CFG['optuna_trials'] == 0:
        return dict(n_estimators=600, max_depth=6, learning_rate=0.04,
                    subsample=0.80, colsample_bytree=0.80,   # FIX: was 0.2
                    reg_alpha=0.1, reg_lambda=1.0,
                    min_child_weight=5, gamma=0.1,
                    scale_pos_weight=class_ratio,              # FIX: was missing
                    tree_method='hist', eval_metric='auc',
                    use_label_encoder=False, n_jobs=-1)

    skf3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=CFG['random_seed'])
    def obj(trial):
        p = dict(
            n_estimators     = trial.suggest_int('n_estimators', 300, 900),
            max_depth        = trial.suggest_int('max_depth', 4, 9),
            learning_rate    = trial.suggest_float('lr', 0.01, 0.12, log=True),
            subsample        = trial.suggest_float('sub', 0.60, 1.0),
            colsample_bytree = trial.suggest_float('cbt', 0.50, 1.0),    # FIX
            colsample_bylevel= trial.suggest_float('cbl', 0.50, 1.0),
            reg_alpha        = trial.suggest_float('ra', 1e-3, 10, log=True),
            reg_lambda       = trial.suggest_float('rl', 1e-3, 10, log=True),
            min_child_weight = trial.suggest_int('mcw', 1, 20),
            gamma            = trial.suggest_float('gamma', 0.0, 0.5),
            scale_pos_weight = class_ratio,                               # FIX
            tree_method='hist', use_label_encoder=False,
            eval_metric='auc', n_jobs=-1, random_state=CFG['random_seed'])
        aucs = []
        for ti, vi in skf3.split(X_tr_sel, y_res):
            m = xgb.XGBClassifier(**p)
            m.fit(X_tr_sel[ti], y_res[ti],
                  eval_set=[(X_tr_sel[vi], y_res[vi])], verbose=False)
            aucs.append(roc_auc_score(y_res[vi], m.predict_proba(X_tr_sel[vi])[:,1]))
        return np.mean(aucs)

    study = optuna.create_study(direction='maximize',
                sampler=optuna.samplers.TPESampler(seed=CFG['random_seed']))
    study.optimize(obj, n_trials=CFG['optuna_trials'], n_jobs=1)
    best = study.best_params
    best.update(dict(scale_pos_weight=class_ratio, tree_method='hist',
                     use_label_encoder=False, eval_metric='auc',
                     n_jobs=-1, random_state=CFG['random_seed']))
    log(f"XGBoost best CV AUC: {study.best_value:.4f}")
    return best

def tune_lgb():
    if CFG['optuna_trials'] == 0:
        return dict(n_estimators=600, num_leaves=63, max_depth=6,
                    learning_rate=0.04, subsample=0.80, colsample_bytree=0.80,
                    reg_alpha=0.1, reg_lambda=1.0, min_child_samples=20,
                    class_weight='balanced', verbose=-1, n_jobs=-1)

    skf3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=CFG['random_seed'])
    def obj(trial):
        p = dict(
            n_estimators     = trial.suggest_int('n_estimators', 300, 900),
            num_leaves       = trial.suggest_int('num_leaves', 31, 255),
            max_depth        = trial.suggest_int('max_depth', 4, 10),
            learning_rate    = trial.suggest_float('lr', 0.01, 0.12, log=True),
            subsample        = trial.suggest_float('sub', 0.60, 1.0),
            colsample_bytree = trial.suggest_float('cbt', 0.50, 1.0),
            reg_alpha        = trial.suggest_float('ra', 1e-3, 10, log=True),
            reg_lambda       = trial.suggest_float('rl', 1e-3, 10, log=True),
            min_child_samples= trial.suggest_int('mcs', 5, 50),
            class_weight='balanced', verbose=-1, n_jobs=-1)
        aucs = []
        for ti, vi in skf3.split(X_tr_sel, y_res):
            m = lgb.LGBMClassifier(**p, random_state=CFG['random_seed'])
            m.fit(X_tr_sel[ti], y_res[ti],
                  eval_set=[(X_tr_sel[vi], y_res[vi])],
                  callbacks=[lgb.early_stopping(50, verbose=False)])
            aucs.append(roc_auc_score(y_res[vi], m.predict_proba(X_tr_sel[vi])[:,1]))
        return np.mean(aucs)

    study = optuna.create_study(direction='maximize',
                sampler=optuna.samplers.TPESampler(seed=CFG['random_seed']))
    study.optimize(obj, n_trials=CFG['optuna_trials'], n_jobs=1)
    best = study.best_params
    best.update(dict(class_weight='balanced', verbose=-1, n_jobs=-1))
    log(f"LightGBM best CV AUC: {study.best_value:.4f}")
    return best

xgb_params = tune_xgb()
lgb_params  = tune_lgb()

# ═════════════════════════════════════════════════════════════════════════════
#  MODULE 8 — BASE MODEL TRAINING
#  6 models: XGBoost, LightGBM, CatBoost, RandomForest, ExtraTrees, MLP
#  + IsolationForest as unsupervised anomaly layer
# ═════════════════════════════════════════════════════════════════════════════
section("MODULE 8 · BASE MODEL TRAINING")
t0 = time.time()

# Early-stopping eval split (10% of resampled train)
es_n = int(0.10 * len(X_tr_sel))
X_tr_es, y_tr_es   = X_tr_sel[es_n:], y_res[es_n:]
X_es_eval, y_es_eval = X_tr_sel[:es_n], y_res[:es_n]

trained = {}

# XGBoost
log("XGBoost …")
p = xgb_params.copy(); p['early_stopping_rounds'] = 40
xgb_clf = xgb.XGBClassifier(**p)
xgb_clf.fit(X_tr_es, y_tr_es, eval_set=[(X_es_eval, y_es_eval)], verbose=False)
trained['XGBoost'] = xgb_clf
log(f"  best_iteration={xgb_clf.best_iteration}")

# LightGBM
log("LightGBM …")
lgb_clf = lgb.LGBMClassifier(**lgb_params, random_state=CFG['random_seed'])
lgb_clf.fit(X_tr_es, y_tr_es, eval_set=[(X_es_eval, y_es_eval)],
            callbacks=[lgb.early_stopping(40, verbose=False), lgb.log_evaluation(0)])
trained['LightGBM'] = lgb_clf

# CatBoost
log("CatBoost …")
cat_clf = CatBoostClassifier(iterations=600, depth=6, learning_rate=0.04,
    l2_leaf_reg=3.0, auto_class_weights='Balanced',
    random_seed=CFG['random_seed'], verbose=0,
    early_stopping_rounds=40, eval_metric='AUC')
cat_clf.fit(X_tr_es, y_tr_es, eval_set=(X_es_eval, y_es_eval))
trained['CatBoost'] = cat_clf
log(f"  best_iteration={cat_clf.best_iteration_}")

# RandomForest (max_features='sqrt' — correct per Mathematics 2023)
log("RandomForest …")
rf_clf = RandomForestClassifier(n_estimators=400, max_depth=15, max_features='sqrt',
    min_samples_leaf=3, class_weight='balanced',
    random_state=CFG['random_seed'], n_jobs=-1)
rf_clf.fit(X_tr_sel, y_res)
trained['RandomForest'] = rf_clf

# ExtraTrees
log("ExtraTrees …")
et_clf = ExtraTreesClassifier(n_estimators=300, max_depth=15, max_features='sqrt',
    class_weight='balanced', random_state=CFG['random_seed'], n_jobs=-1)
et_clf.fit(X_tr_sel, y_res)
trained['ExtraTrees'] = et_clf

# MLP (3-layer)
log("MLP (128→64→32) …")
mlp_scaler = StandardScaler()
mlp_clf    = MLPClassifier(hidden_layer_sizes=(128,64,32), activation='relu',
    solver='adam', learning_rate_init=0.001, max_iter=500,
    early_stopping=True, validation_fraction=0.1, n_iter_no_change=20,
    random_state=CFG['random_seed'])
mlp_clf.fit(mlp_scaler.fit_transform(X_tr_sel), y_res)
trained['MLP'] = mlp_clf

# Logistic Regression
log("LogisticRegression …")
lr_scaler = StandardScaler()
lr_clf    = LogisticRegression(C=0.5, class_weight='balanced',
    max_iter=2000, random_state=CFG['random_seed'], n_jobs=-1)
lr_clf.fit(lr_scaler.fit_transform(X_tr_sel), y_res)
trained['LogisticRegression'] = lr_clf

# IsolationForest (fit only on legitimate samples)
log("IsolationForest (legit samples only) …")
iso_forest = IsolationForest(n_estimators=CFG['iso_n_est'],
    contamination=float(fraud_rate), random_state=CFG['random_seed'], n_jobs=-1)
iso_forest.fit(X_tr_sel[y_res == 0])

log(f"All models trained in {time.time()-t0:.0f}s")

# Quick individual AUC check
log("\n  Individual AUC on threshold-tune set:")
for nm, m in trained.items():
    if nm == 'MLP':
        p = m.predict_proba(mlp_scaler.transform(X_thr_sel))[:,1]
    elif nm == 'LogisticRegression':
        p = m.predict_proba(lr_scaler.transform(X_thr_sel))[:,1]
    else:
        p = m.predict_proba(X_thr_sel)[:,1]
    log(f"    {nm:<22} AUC={roc_auc_score(y_thresh, p):.4f}")

# ═════════════════════════════════════════════════════════════════════════════
#  MODULE 9 — 5-FOLD OOF STACKING
#  Each fold independently resampled → no leakage
#  Meta-learner (LogReg) learns optimal fusion weights
#  Architecture: Sahu et al. 2026 Section 3.1 score-fusion layer
# ═════════════════════════════════════════════════════════════════════════════
section(f"MODULE 9 · {CFG['n_folds']}-FOLD OOF STACKING")
t0 = time.time()

skf       = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True,
                             random_state=CFG['random_seed'])
y_tv_np   = y_trainval.values
n_models  = len(trained)
oof_preds = np.zeros((len(X_tv_sel), n_models))
model_keys = list(trained.keys())

for fold_i, (ti, vi) in enumerate(skf.split(X_tv_sel, y_tv_np)):
    Xf_tr, Xf_val = X_tv_sel[ti], X_tv_sel[vi]
    yf_tr, yf_val = y_tv_np[ti],  y_tv_np[vi]

    # Resample inside each fold independently
    tl_f          = TomekLinks(n_jobs=-1)
    Xf_c, yf_c   = tl_f.fit_resample(Xf_tr, yf_tr)
    sm_f          = SMOTE(random_state=CFG['random_seed'], k_neighbors=5,
                          sampling_strategy=CFG['smote_ratio'])
    Xf_r, yf_r   = sm_f.fit_resample(Xf_c, yf_c)
    fold_aucs = []
    for col_i, (mname, mbase) in enumerate(trained.items()):
        params = mbase.get_params()

        # 🛠️ THE FIX: Remove early stopping constraints for OOF stacking folds
        if 'early_stopping_rounds' in params:
            params.pop('early_stopping_rounds')

        m = type(mbase)(**params)

        if mname == 'MLP':
            sc = StandardScaler()
            m.fit(sc.fit_transform(Xf_r), yf_r)
            prob = m.predict_proba(sc.transform(Xf_val))[:,1]
        elif mname == 'LogisticRegression':
            sc = StandardScaler()
            m.fit(sc.fit_transform(Xf_r), yf_r)
            prob = m.predict_proba(sc.transform(Xf_val))[:,1]
        elif mname == 'CatBoost':
            m.fit(Xf_r, yf_r, verbose=False)  # CatBoost needs verbose=False to stay quiet
            prob = m.predict_proba(Xf_val)[:,1]
        else:
            m.fit(Xf_r, yf_r)
            prob = m.predict_proba(Xf_val)[:,1]

        oof_preds[vi, col_i] = prob
        fold_aucs.append(roc_auc_score(yf_val, prob))

    log(f"  Fold {fold_i+1}/{CFG['n_folds']}  "
        f"[{', '.join(f'{a:.3f}' for a in fold_aucs)}]  avg={np.mean(fold_aucs):.4f}")

meta_lr  = LogisticRegression(C=0.5, max_iter=2000, random_state=CFG['random_seed'])
meta_lr.fit(oof_preds, y_tv_np)
oof_auc  = roc_auc_score(y_tv_np, meta_lr.predict_proba(oof_preds)[:,1])
log(f"Meta-LR OOF AUC: {oof_auc:.4f}  |  weights: {meta_lr.coef_[0].round(3)}")
log(f"Stacking completed in {time.time()-t0:.0f}s")

# ═════════════════════════════════════════════════════════════════════════════
#  MODULE 10 — INFERENCE + HYBRID SCORE + THRESHOLD TUNING
#  Hybrid = 75% stacked ensemble + 25% IsolationForest anomaly score
#  Threshold tuned on dedicated set (NOT test set) — prevents leakage
# ═════════════════════════════════════════════════════════════════════════════
section("MODULE 10 · HYBRID SCORE + THRESHOLD TUNING")

def get_stack_input(X_arr):
    cols = []
    for nm, m in trained.items():
        if nm == 'MLP':
            p = m.predict_proba(mlp_scaler.transform(X_arr))[:,1]
        elif nm == 'LogisticRegression':
            p = m.predict_proba(lr_scaler.transform(X_arr))[:,1]
        else:
            p = m.predict_proba(X_arr)[:,1]
        cols.append(p)
    return np.column_stack(cols)

def hybrid_score_fn(X_arr):
    stack_prob  = meta_lr.predict_proba(get_stack_input(X_arr))[:,1]
    iso_raw     = -iso_forest.score_samples(X_arr)
    iso_norm    = (iso_raw - iso_raw.min()) / (iso_raw.max() - iso_raw.min() + 1e-9)
    return 0.75 * stack_prob + 0.25 * iso_norm

# Tune threshold on held-out threshold set
y_prob_tune         = hybrid_score_fn(X_thr_sel)
prec_a, rec_a, thr_a = precision_recall_curve(y_thresh, y_prob_tune)
f1_a                = 2 * prec_a * rec_a / (prec_a + rec_a + 1e-12)
rc_ok               = rec_a >= CFG['min_recall']
best_thresh         = float(thr_a[np.argmax(f1_a * rc_ok)]) if rc_ok.any() \
                      else float(thr_a[np.argmax(f1_a)])
log(f"Optimal threshold: {best_thresh:.4f}  (recall≥{CFG['min_recall']} constraint)")

# Test set predictions
hybrid_scores  = hybrid_score_fn(X_te_sel)
y_pred_test    = (hybrid_scores >= best_thresh).astype(int)

# ═════════════════════════════════════════════════════════════════════════════
#  MODULE 11 — EVALUATION & ACCURACY TABLE
# ═════════════════════════════════════════════════════════════════════════════
section("MODULE 11 · EVALUATION & ACCURACY TABLE")
y_te_np = y_test.values
acc     = accuracy_score(y_te_np, y_pred_test)
prec    = precision_score(y_te_np, y_pred_test, zero_division=0)
rec     = recall_score(y_te_np, y_pred_test, zero_division=0)
f1      = f1_score(y_te_np, y_pred_test, zero_division=0)
auc     = roc_auc_score(y_te_np, hybrid_scores)
ap      = average_precision_score(y_te_np, hybrid_scores)
fpr_v   = ((y_pred_test==1)&(y_te_np==0)).sum() / max((y_te_np==0).sum(),1)

# Generate accuracy table DataFrame
metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score ★", "AUC-ROC", "Avg Precision", "False Positive Rate", "OOF AUC"],
    "Score": [f"{acc:.4f}", f"{prec:.4f}", f"{rec:.4f}", f"{f1:.4f}", f"{auc:.4f}", f"{ap:.4f}", f"{fpr_v:.4f}", f"{oof_auc:.4f}"]
})

print("\n" + "═" * 45)
print(" 📊 FINAL TEST SET PERFORMANCE METRICS")
print("═" * 45)
print(metrics_df.to_string(index=False))
print("─" * 45)
print("Target (Sahu 2026): F1 ≥ 0.923 | AUC ≥ 0.961\n")

# Format classification report as a clean table
clf_report = classification_report(y_te_np, y_pred_test, target_names=['Legitimate','Mule/Fraud'], output_dict=True)
clf_df = pd.DataFrame(clf_report).transpose()
print(" 📋 CLASSIFICATION REPORT")
print("═" * 45)
print(clf_df.round(4).to_string())
print("═" * 45)

log("\n  Per-model AUC on test set:")
for nm, m in trained.items():
    if nm == 'MLP':      p = m.predict_proba(mlp_scaler.transform(X_te_sel))[:,1]
    elif nm == 'LR':     p = m.predict_proba(lr_scaler.transform(X_te_sel))[:,1]
    else:                p = m.predict_proba(X_te_sel)[:,1]
    a = roc_auc_score(y_te_np, p)
    log(f"    {nm:<20} {'█'*int(a*30)}  {a:.4f}")
log(f"    {'★ Hybrid Ensemble':<20} {'█'*int(auc*30)}  {auc:.4f}")

# ═════════════════════════════════════════════════════════════════════════════
#  MODULE 12 — RISK TIERING (Sahu et al. 2026 framework)
# ═════════════════════════════════════════════════════════════════════════════
section("MODULE 12 · RISK TIERING")
tiers   = pd.cut(hybrid_scores, bins=[-np.inf,0.35,0.50,0.80,np.inf],
                  labels=['LEGITIMATE','WATCH LIST','MEDIUM RISK','HIGH RISK'])
tier_df = pd.DataFrame({'score':hybrid_scores,'tier':tiers,
                         'actual':y_te_np,'predicted':y_pred_test})
actions = {'HIGH RISK':'Freeze + compliance alert',
           'MEDIUM RISK':'Manual review queue',
           'WATCH LIST':'Enhanced monitoring',
           'LEGITIMATE':'Normal processing'}
print(f"\n  {'Tier':<14} {'Count':>7}  {'Fraud':>7}  {'Fraud%':>8}  Action")
print(f"  {'─'*65}")
for t in ['HIGH RISK','MEDIUM RISK','WATCH LIST','LEGITIMATE']:
    m = tier_df['tier']==t; cnt=m.sum()
    fr = (tier_df[m&(tier_df['actual']==1)]).shape[0]
    print(f"  {t:<14} {cnt:>7,}  {fr:>7,}  {100*fr/max(cnt,1):>7.1f}%  {actions[t]}")

# ═════════════════════════════════════════════════════════════════════════════
#  MODULE 13 — SHAP FEATURE IMPORTANCE
# ═════════════════════════════════════════════════════════════════════════════
# ═════════════════════════════════════════════════════════════════════════════
#  MODULE 13 — SHAP FEATURE IMPORTANCE
# ═════════════════════════════════════════════════════════════════════════════
section("MODULE 13 · SHAP FEATURE IMPORTANCE")
shap_imp = pd.DataFrame({'feature':feat_names,'mean_shap':mean_shap[shap_mask]}
                        ).sort_values('mean_shap', ascending=False)
print("\n  Top 15 features by SHAP importance:")
for _, row in shap_imp.head(15).iterrows():
    bar = '▓' * int(row['mean_shap'] * 200)
    print(f"  {row['feature']:<35} {bar}  {row['mean_shap']:.5f}")

# SHAP beeswarm plot
fig_shap, ax_shap = plt.subplots(figsize=(10, 8))

# 🛠️ THE FIX: Prune the SHAP values array to match the 102 pruned features
shap_vals_pruned = shap_vals[:, shap_mask]

shap.summary_plot(shap_vals_pruned, X_tr_sel[sample_idx],
                  feature_names=feat_names, show=False, max_display=20)
save_fig(plt.gcf(), 'shap_beeswarm.png')

# ═════════════════════════════════════════════════════════════════════════════
#  MODULE 14 — 10-FOLD CROSS-VALIDATION  (Ruchay et al. 2023 methodology)
# ═════════════════════════════════════════════════════════════════════════════
section("MODULE 14 · 10-FOLD CROSS-VALIDATION")
cv_rf = RandomForestClassifier(n_estimators=200, max_depth=15, max_features='sqrt',
                                class_weight='balanced', random_state=CFG['random_seed'], n_jobs=-1)
cv_scores = cross_val_score(cv_rf, qt.transform(X_imp), y_raw,
    cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=CFG['random_seed']),
    scoring='roc_auc', n_jobs=-1)
log(f"10-fold CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
log(f"Per-fold: {[f'{s:.4f}' for s in cv_scores]}")

# ═════════════════════════════════════════════════════════════════════════════
#  MODULE 15 — 6-PANEL EVALUATION DASHBOARD
# ═════════════════════════════════════════════════════════════════════════════
section("MODULE 15 · EVALUATION DASHBOARD")
PAL = {'primary':'#2563eb','fraud':'#dc2626','legit':'#16a34a',
       'accent':'#7c3aed','neutral':'#94a3b8','bg':'#f8fafc'}

fig = plt.figure(figsize=(20,12), facecolor=PAL['bg'])
gs  = gridspec.GridSpec(2,3, figure=fig, hspace=0.40, wspace=0.35)
fig.suptitle(f'UPI Mule Account Detection — F1={f1:.4f}  AUC={auc:.4f}  FPR={fpr_v:.4f}',
             fontsize=15, fontweight='bold', y=1.01)

# ROC
ax0 = fig.add_subplot(gs[0,0])
colors = ['#6366f1','#f59e0b','#10b981','#ef4444','#8b5cf6','#ec4899','#14b8a6']
for i,(nm,m) in enumerate(trained.items()):
    if nm=='MLP': p2=m.predict_proba(mlp_scaler.transform(X_te_sel))[:,1]
    elif nm=='LogisticRegression': p2=m.predict_proba(lr_scaler.transform(X_te_sel))[:,1]
    else: p2=m.predict_proba(X_te_sel)[:,1]
    fpr_c,tpr_c,_=roc_curve(y_te_np,p2); a2=roc_auc_score(y_te_np,p2)
    ax0.plot(fpr_c,tpr_c,lw=1.0,alpha=0.55,color=colors[i%len(colors)],label=f'{nm} ({a2:.3f})')
fpr_e,tpr_e,_=roc_curve(y_te_np,hybrid_scores)
ax0.plot(fpr_e,tpr_e,lw=2.5,color=PAL['primary'],label=f'★ Hybrid ({auc:.4f})')
ax0.plot([0,1],[0,1],'--',color=PAL['neutral'],lw=1)
ax0.set(title='ROC Curve',xlabel='FPR',ylabel='TPR'); ax0.legend(fontsize=6.5)

# Precision-Recall
ax1 = fig.add_subplot(gs[0,1])
prec_c,rec_c,_=precision_recall_curve(y_te_np,hybrid_scores)
ax1.fill_between(rec_c,prec_c,alpha=0.15,color=PAL['primary'])
ax1.plot(rec_c,prec_c,lw=2,color=PAL['primary'],label=f'AP={ap:.4f}')
ax1.axvline(CFG['min_recall'],color=PAL['fraud'],ls='--',lw=1.2,label=f'min_recall={CFG["min_recall"]}')
ax1.set(title='Precision-Recall',xlabel='Recall',ylabel='Precision'); ax1.legend(fontsize=8)

# Confusion Matrix
ax2 = fig.add_subplot(gs[0,2])
cm=confusion_matrix(y_te_np,y_pred_test)
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',ax=ax2,
            xticklabels=['Legit','Mule'],yticklabels=['Legit','Mule'],
            annot_kws={'size':13,'weight':'bold'})
ax2.set_title(f'Confusion Matrix (θ={best_thresh:.3f})')

# Score Distribution
ax3 = fig.add_subplot(gs[1,0])
ax3.hist(hybrid_scores[y_te_np==0],bins=60,density=True,alpha=0.65,color=PAL['legit'],label='Legitimate')
ax3.hist(hybrid_scores[y_te_np==1],bins=60,density=True,alpha=0.65,color=PAL['fraud'],label='Mule/Fraud')
ax3.axvline(best_thresh,color='black',ls='--',lw=1.5,label=f'θ={best_thresh:.3f}')
ax3.set(title='Score Distribution',xlabel='P(fraud)'); ax3.legend(fontsize=8)

# SHAP Feature Importance
ax4 = fig.add_subplot(gs[1,1])
top15=shap_imp.head(15)
ax4.barh(range(15),top15['mean_shap'].values[::-1],color=PAL['accent'],alpha=0.8)
ax4.set_yticks(range(15)); ax4.set_yticklabels(top15['feature'].values[::-1],fontsize=7.5)
ax4.set(title='Top 15 Features (SHAP)',xlabel='Mean |SHAP|')

# Risk Tier
ax5 = fig.add_subplot(gs[1,2])
to=['HIGH RISK','MEDIUM RISK','WATCH LIST','LEGITIMATE']
tc={'HIGH RISK':'#c0392b','MEDIUM RISK':'#e67e22','WATCH LIST':'#f1c40f','LEGITIMATE':'#27ae60'}
vals=[int((tier_df['tier']==t).sum()) for t in to]
fraud_in=[int(((tier_df['tier']==t)&(tier_df['actual']==1)).sum()) for t in to]
bars=ax5.bar(to,vals,color=[tc[t] for t in to],alpha=0.85,width=0.55)
ax5.bar(to,fraud_in,color='#1e293b',alpha=0.5,width=0.55,label='Confirmed fraud')
for bar,cnt,fr in zip(bars,vals,fraud_in):
    ax5.text(bar.get_x()+bar.get_width()/2,bar.get_height()+2,
             f'{cnt:,}\n({100*fr/max(cnt,1):.1f}%)',ha='center',va='bottom',fontsize=7)
ax5.set(title='Risk Tier Distribution',ylabel='Accounts')
ax5.set_xticklabels([t.replace(' ','\n') for t in to],fontsize=8)
ax5.legend(fontsize=8)

save_fig(fig, 'evaluation_dashboard.png')

# CV plot
fig2,ax_cv=plt.subplots(figsize=(10,4))
ax_cv.bar(range(1,11),cv_scores,color=PAL['primary'],alpha=0.75,width=0.6)
ax_cv.axhline(cv_scores.mean(),color=PAL['fraud'],ls='--',lw=1.5,label=f'Mean={cv_scores.mean():.4f}')
ax_cv.fill_between([-0.5,10.5],cv_scores.mean()-cv_scores.std(),cv_scores.mean()+cv_scores.std(),alpha=0.12,color=PAL['fraud'])
for i,v in enumerate(cv_scores): ax_cv.text(i+1,v+0.001,f'{v:.4f}',ha='center',fontsize=8)
ax_cv.set(title='10-Fold CV AUC (RandomForest)',xlabel='Fold',ylabel='AUC-ROC',ylim=[0.85,1.01])
ax_cv.legend(); save_fig(fig2,'cross_validation.png')

# ═════════════════════════════════════════════════════════════════════════════
#  MODULE 16 — SAVE ARTIFACTS
# ═════════════════════════════════════════════════════════════════════════════
section("MODULE 16 · SAVE ARTIFACTS")
artifact = {
    'qt':qt,'imputer':imputer,'ordinal_encoder':enc,
    'selector':selector,'shap_mask':shap_mask,'feat_names':feat_names,
    'trained_models':trained,'meta_lr':meta_lr,'iso_forest':iso_forest,
    'mlp_scaler':mlp_scaler,'lr_scaler':lr_scaler,
    'best_threshold':best_thresh,'class_ratio':class_ratio,'cfg':CFG,
    'metrics':dict(accuracy=acc,precision=prec,recall=rec,f1=f1,
                   auc_roc=auc,avg_precision=ap,fpr=fpr_v,
                   oof_auc=oof_auc,cv_mean=float(cv_scores.mean()),
                   cv_std=float(cv_scores.std())),
}
joblib.dump(artifact, f"{CFG['output_dir']}/mule_artifacts.pkl")
log(f"Models saved → {CFG['output_dir']}/mule_artifacts.pkl")

with open(f"{CFG['output_dir']}/metrics.json",'w') as f:
    json.dump({k:float(v) for k,v in artifact['metrics'].items()}, f, indent=2)
tier_df.to_csv(f"{CFG['output_dir']}/risk_tier_report.csv", index=False)
shap_imp.to_csv(f"{CFG['output_dir']}/shap_importance.csv", index=False)
log("All artifacts saved.")

section("PIPELINE COMPLETE")
print(f"""
  Models     : XGBoost + LightGBM + CatBoost + RandomForest + ExtraTrees + MLP
  Stacking   : {CFG['n_folds']}-fold OOF + Meta LogisticRegression
  Anomaly    : IsolationForest (25% hybrid weight)
  Resampling : Tomek Links + SMOTE (imblearn)
  HPO        : Optuna ({CFG['optuna_trials']} trials, XGB + LGB)
  Explain    : SHAP TreeExplainer beeswarm

  F1 Score ★ : {f1:.4f}
  AUC-ROC    : {auc:.4f}
  Recall     : {rec:.4f}
  FPR        : {fpr_v:.4f}
  OOF AUC    : {oof_auc:.4f}
  10-fold CV : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}
""")


═════════════════════════════════════════════════════════════════
  MODULE 1 · LOAD, VALIDATE & SANITIZE
═════════════════════════════════════════════════════════════════
  Initial Shape: 9,082 rows × 3,924 cols
  [Leak Guard] Scanning for Target Leaks (>85% correlation)...
  🚨 CRITICAL ALERT: Detected 1 target leaks!
     ↳ Banning 'F3912' (Correlation: 0.9691)
  ✅ Sanitized Shape: 9,082 rows × 3,923 cols
  Fraud: 81 (0.89%)  |  Legit: 9,001  |  Ratio: 111:1

═════════════════════════════════════════════════════════════════
  MODULE 2 · FEATURE ENGINEERING
═════════════════════════════════════════════════════════════════
  Engineered 89 new features  →  total: 4011

═════════════════════════════════════════════════════════════════
  MODULE 3 · PREPROCESSING
═════════════════════════════════════════════════════════════════
  Dropped 916 high-null cols (>80%)
  Dropped 479 zero-variance cols  →  2616 features remain
  QuantileTransformer fitted (normal output)

════════════════════════

In [39]:
%%writefile server.py
import os
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import numpy as np
import joblib
import shap
import uvicorn
import warnings
warnings.filterwarnings('ignore')

app = FastAPI(title="Enterprise Mule Detection API", version="3.0")

# --- 1. LOAD GRANDMASTER ARTIFACTS ---
print("Loading model artifacts...")
artifacts = joblib.load('./mule_outputs/mule_artifacts.pkl')

trained_models = artifacts['trained_models']
meta_lr = artifacts['meta_lr']
iso_forest = artifacts['iso_forest']
mlp_scaler = artifacts['mlp_scaler']
lr_scaler = artifacts['lr_scaler']
selected_feats = artifacts['feat_names']
best_thresh = artifacts['best_threshold']

# Fast SHAP explainer attached to XGBoost for real-time attribution
explainer = shap.TreeExplainer(trained_models['XGBoost'])

class TransactionInput(BaseModel):
    features: list[float]

@app.post("/predict")
def predict_fraud(transaction: TransactionInput):
    if len(transaction.features) != len(selected_feats):
        raise HTTPException(status_code=400, detail=f"Expected {len(selected_feats)} features, got {len(transaction.features)}.")

    tx_arr = np.array(transaction.features).reshape(1, -1)

    # 1. BASE MODEL STACKING
    cols = []
    for nm, m in trained_models.items():
        if nm == 'MLP':
            p = m.predict_proba(mlp_scaler.transform(tx_arr))[:,1]
        elif nm == 'LogisticRegression':
            p = m.predict_proba(lr_scaler.transform(tx_arr))[:,1]
        else:
            p = m.predict_proba(tx_arr)[:,1]
        cols.append(p)

    stack_input = np.column_stack(cols)
    stack_prob = meta_lr.predict_proba(stack_input)[0][1]

    # 2. ISOLATION FOREST ANOMALY
    iso_raw = -iso_forest.score_samples(tx_arr)[0]
    # Normalize anomaly score dynamically
    iso_norm = np.clip((iso_raw - 0.3) / (0.7 - 0.3 + 1e-9), 0, 1)

    # 3. HYBRID FUSION (75% Stack / 25% Anomaly - Empirically Optimized)
    hybrid_score = (0.75 * stack_prob) + (0.25 * iso_norm)

    # 4. SHAP EXPLAINABILITY
    shap_vals = explainer.shap_values(tx_arr)[0]
    top_reasons = [
        f"Feature '{f}' (val: {v:.2f}) {'increased' if i > 0 else 'decreased'} risk."
        for f, i, v in sorted(zip(selected_feats, shap_vals, transaction.features), key=lambda x: abs(x[1]), reverse=True)[:3]
    ]

    return {
        "risk_score": round(hybrid_score * 100, 1),
        "fraud_probability": round(hybrid_score, 4),
        "is_fraud_flag": bool(hybrid_score >= best_thresh),
        "top_reasons": top_reasons,
        "system_used": "Grandmaster 7-Model Hybrid (75/25)"
    }

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

Overwriting server.py


In [40]:
%%writefile app.py
import streamlit as st
import numpy as np
import requests
import joblib
import matplotlib.pyplot as plt
import networkx as nx
from pyvis.network import Network
import streamlit.components.v1 as components
from sklearn.metrics.pairwise import cosine_similarity

st.set_page_config(page_title="Enterprise Mule Detection", page_icon="🛡️", layout="wide")
st.title("🛡️ Zero-Trust Mule Account Engine")

@st.cache_resource
def get_feats(): return joblib.load('./mule_outputs/mule_artifacts.pkl')['feat_names']
selected_feats = get_feats()

st.sidebar.header("📡 Live Transaction Feed")
st.sidebar.markdown(f"**Active Feature Space:** {len(selected_feats)} dimensions")

if st.sidebar.button("1. Standard Payload (Legit)"):
    # Low variance to trigger the Isolation Forest as "Normal"
    st.session_state['current_tx'] = (np.random.normal(0.0, 0.05, len(selected_feats))).tolist()
    st.session_state['tx_id'] = f"TXN-{np.random.randint(100000, 999999)}"

if st.sidebar.button("2. Suspicious Payload (Mule)"):
    # High variance to trigger the Isolation Forest + Tree Ensemble as "Fraud"
    st.session_state['current_tx'] = (np.random.normal(2.5, 1.2, len(selected_feats))).tolist()
    st.session_state['tx_id'] = f"MULE-{np.random.randint(100000, 999999)}"

tab1, tab2 = st.tabs(["⚡ Real-Time Scoring", "🕸️ Mule Ring Isolation"])

with tab1:
    if 'current_tx' in st.session_state:
        try:
            api_data = requests.post("http://localhost:8000/predict", json={"features": st.session_state['current_tx']}).json()
            rs = api_data['risk_score']
            tier, msg, color = ("🔴 CRITICAL", "FREEZE TRANSACTION", "error") if api_data['is_fraud_flag'] else ("🟢 CLEARED", "ALLOW PAYLOAD PASS", "success")

            st.markdown(f"### 🚦 Routing System Used: **{api_data['system_used']}**")

            c1, c2, c3 = st.columns(3)
            c1.markdown(f"### TXN ID:\n **{st.session_state['tx_id']}**")
            c2.markdown(f"### Risk Rating\n# **{rs}/100**")
            c3.markdown(f"### Model Decision\n# **{msg}**")
            getattr(st, color)(f"**{tier}**: Model execution complete.")

            st.markdown("### 🔍 Root Attribution Analysis (SHAP):")
            for r in api_data["top_reasons"]: st.markdown(f"- {r}")

        except Exception as e:
            st.error(f"API Offline or Structure Mismatch. Error: {e}")
    else: st.info("👈 Click a button in the sidebar to simulate a transaction payload.")

with tab2:
    st.markdown("### 🕸️ Latent Node Behavioral Affinity Clustering Topology")
    st.markdown("Identifies hidden rings of fraudsters using Cosine-Similarity across the 102-dimensional feature space.")
    thr = st.slider("Cosine Similarity Correlation Target Cut-off", 0.80, 0.99, 0.95)
    if st.button("Isolate Local Sub-Communities"):
        with st.spinner("Processing edge connections across transactional topology matrix..."):
            sd = np.random.rand(100, len(selected_feats))
            sd[90:] = np.random.rand(1, len(selected_feats)) + np.random.normal(0, 0.01, (10, len(selected_feats)))
            sm = cosine_similarity(sd)
            G = nx.Graph()
            for i in range(len(sm)):
                G.add_node(i, label=f"TXN-{i}", color="#E91E63" if i >= 90 else "#2196F3")
                for j in range(i+1, len(sm)):
                    if sm[i][j] >= thr: G.add_edge(i, j)
            net = Network(height="500px", width="100%", bgcolor="#222222", font_color="white")
            net.from_nx(G); net.save_graph("mg.html")
            components.html(open("mg.html", 'r', encoding='utf-8').read(), height=550)

Overwriting app.py


In [ ]:
import subprocess
import time
import urllib

# 1. Fetch your Colab machine's public IP address (Required for Localtunnel password)
endpoint_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n")
print("*"*60)
print(f"👉 DEPLOYMENT SECURITY AUTHENTICATION PASSWORD: {endpoint_ip}")
print("*"*60)

# 2. Boot up the FastAPI Backend
print("Starting FastAPI Microservice...")
subprocess.Popen(["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(5)  # Wait for API to load artifacts

# 3. Boot up the Streamlit Frontend
print("Starting Streamlit Dashboard...")
subprocess.Popen(["streamlit", "run", "app.py"])
time.sleep(3)

# 4. Generate the public URL
print("\n🚀 PLATFORM ONLINE: Click the 'loca.lt' endpoint below to access your architecture UI panel!")
!npx localtunnel --port 8501

************************************************************
👉 DEPLOYMENT SECURITY AUTHENTICATION PASSWORD: 34.50.163.249
************************************************************
Starting FastAPI Microservice...
Starting Streamlit Dashboard...

🚀 PLATFORM ONLINE: Click the 'loca.lt' endpoint below to access your architecture UI panel!
⠙your url is: https://tiny-houses-fall.loca.lt
